# Двухбашенный кандген: обучение на GPU (Google Colab или DataSphere)

Как устроено — `docs/twotower.md`. Ноутбук только запускает скрипты из корня репозитория:
`scripts/train_twotower.py` (обучение) и `scripts/encode_twotower.py` (векторы треков и запросов).
Параметры — `configs/twotower.yaml`, любой переопределяется через `--set`.

**Colab:** Runtime → Change runtime type → GPU (лучше A100; на T4 обучение в разы дольше и в fp16).
Данных нет в git — положите их на Google Drive в папку `MyDrive/recsys/`:

```
MyDrive/recsys/
  music4all_crs/              tracks_meta.parquet, train.parquet, train_queries.parquet, train_qrels.parquet,
                              test_public.parquet, test_public_queries.parquet, test_public_qrels.parquet
  artifacts/semantic_ids/     semantic_codes.npy, rqvae_checkpoint.pt
```

Результаты пишутся в `MyDrive/recsys/artifacts/twotower/` — переживают отключение Colab.
**DataSphere:** откройте ноутбук из корня репозитория, первая ячейка ничего не делает.

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/rimgro/conversational-recsys.git"
BRANCH = "dev/two-tower"
DRIVE_DIR = "/content/drive/MyDrive/recsys"      # данные и результаты на Google Drive
REPO_DIR = "/content/conversational-recsys"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    url = REPO_URL
    try:                                         # приватный репозиторий: секрет GITHUB_TOKEN (ключик слева)
        from google.colab import userdata
        url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:
        pass
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-b", BRANCH, url, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run(["git", "pull", "--ff-only"], check=True)
    need = [f"music4all_crs/{n}.parquet" for n in ("tracks_meta", "train", "train_queries", "train_qrels",
            "test_public", "test_public_queries", "test_public_qrels")]
    need += ["artifacts/semantic_ids/semantic_codes.npy", "artifacts/semantic_ids/rqvae_checkpoint.pt"]
    missing = [p for p in need if not os.path.exists(f"{DRIVE_DIR}/{p}")]
    if missing:
        raise FileNotFoundError("на Google Drive нет файлов (загрузите их в эти папки и запустите ячейку снова):\n  "
                                + "\n  ".join(f"{DRIVE_DIR}/{p}" for p in missing))
    # датасет — на локальный диск VM (с Drive чтение медленное); artifacts — ссылкой на Drive (результаты сохранятся)
    subprocess.run(["rsync", "-a", f"{DRIVE_DIR}/music4all_crs/", "music4all_crs/"], check=True)
    os.makedirs(f"{DRIVE_DIR}/artifacts/twotower", exist_ok=True)
    if not os.path.islink("artifacts"):
        os.symlink(f"{DRIVE_DIR}/artifacts", "artifacts")
print("папка:", os.getcwd(), "| ветка:", subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"],
      capture_output=True, text=True).stdout.strip())

In [ ]:
# пакеты для обучения (torch в Colab и на GPU-конфигурации DataSphere уже стоит)
%pip install -q "transformers>=4.51" "peft>=0.13" pyyaml pyarrow
# torchao, предустановленный в Colab, старый: новая peft на нём падает при создании LoRA (нам он не нужен)
%pip uninstall -y -q torchao


In [ ]:
import torch
assert os.path.exists("scripts/train_twotower.py"), "откройте ноутбук из корня репозитория"
for p in ["music4all_crs/train.parquet", "music4all_crs/train_queries.parquet", "music4all_crs/tracks_meta.parquet",
          "music4all_crs/test_public.parquet", "artifacts/semantic_ids/semantic_codes.npy",
          "artifacts/semantic_ids/rqvae_checkpoint.pt"]:
    print(("ok  " if os.path.exists(p) else "НЕТ ") + p)
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print("GPU:", torch.cuda.get_device_name(0), "| точность:", "bf16" if cap[0] >= 8 else "fp16 (bf16 нет)")
else:
    print("GPU нет: Runtime -> Change runtime type -> GPU")

In [ ]:
# коды и чекпоинт RQ-VAE согласованы с датасетом (должно быть 100%)
!python scripts/check_semantic_ids.py

## 1. Быстрая проверка (минуты)
300 пользователей, по 20 шагов на этап: Qwen скачивается, всё грузится и считается. По времени шага в логе видно,
сколько займёт полное обучение (шагов на этапе B ≈ 4250 при батче 128).

In [ ]:
!python scripts/train_twotower.py --run-dir artifacts/twotower/smoke_v2 --n-users 300 --set twotower.train.max_steps=20

## 2. Основная модель (полное обучение, arch v2)
Коды + теги, общие таблицы башен, короткий путь из истории (`docs/twotower.md`); 2 эпохи (~5–6 ч на A100).
В логе «пусто» — средний вес «ничего из истории». Первая модель (v1) лежит в `artifacts/twotower/main`.
Этап A (Qwen заморожен) и этап B (+ LoRA на attention). В выводе: раз в 50 шагов — loss, секунд на шаг и
сколько осталось до конца этапа; раз в 500 шагов — валидация (recall) и сохранение на Drive:
`model.pt` (лучшая модель) и `checkpoint.pt` (для продолжения). Лог целиком — `artifacts/twotower/v2/log.jsonl`.

**Сессия оборвалась** — запустить ячейки 1–3 и эту ячейку снова: `--resume` продолжит с последнего
`checkpoint.pt` (потеряется не больше 500 шагов). Законченное обучение `--resume` не повторяет.

Не хватает памяти GPU — `--set twotower.train.batch_size=64 --set twotower.train.grad_accum=2`.

In [ ]:
!python scripts/train_twotower.py --resume --run-dir artifacts/twotower/v2

In [ ]:
import json
m = json.load(open("artifacts/twotower/v2/metrics.json"))
print("лучшее на валидации:", m["best_val"])
{t: round(v["recall@100"], 3) for t, v in m["last_val"]["by_type"].items()}

## 3. Векторы для пайплайна
Треки и все запросы test_public -> `artifacts/twotower/v2/` (+ `recall_test_public.json`: recall по всему каталогу,
по типам). Дальше GPU не нужен: папку можно скачать и считать метрики где угодно.

In [ ]:
!python scripts/encode_twotower.py --run-dir artifacts/twotower/v2

In [ ]:
# ablation длины истории без переобучения
for L in (0, 16):
    !python scripts/encode_twotower.py --run-dir artifacts/twotower/v2 --history-len {L} --out artifacts/twotower/v2/h{L}

In [ ]:
# только этот кандген (кандгенов-соседей не включаем) на 1000 пользователях test_public
!python scripts/evaluate_candgen.py --n-users 1000 --sources twotower --set retrieval.twotower.dir=artifacts/twotower/v2

In [ ]:
# примеры: запрос -> цель и top-10 модели (артист — название); --type mood — только этот тип запросов
!python scripts/show_twotower.py --run-dir artifacts/twotower/v2 -n 5
!python scripts/show_twotower.py --run-dir artifacts/twotower/v2 --type similar_to -n 3


## 4. Ablation
Каждый — своя папка; дальше так же `encode_twotower.py` и `evaluate_candgen.py`. Порядок — по пользе.

In [ ]:
# без LoRA: только этап A, столько же шагов
!python scripts/train_twotower.py --resume --run-dir artifacts/twotower/no_lora --set 'twotower.train.stages=[{name: A, epochs: 1.3, lora: false, lr: 1.0e-3}]'

In [ ]:
# случайные коды вместо RQ-VAE (semantic ID против хеш-ID)
!python scripts/train_twotower.py --resume --run-dir artifacts/twotower/random_codes --set twotower.codes.source=random --set twotower.codes.init=random

In [ ]:
# in-batch негативы без и с поправкой logQ
!python scripts/train_twotower.py --resume --run-dir artifacts/twotower/in_batch --set twotower.loss.negatives=in_batch
!python scripts/train_twotower.py --resume --run-dir artifacts/twotower/in_batch_logq --set twotower.loss.negatives=in_batch --set twotower.loss.logq=true